# PM10 & Pediatric Asthma in Davao City (2013–2022)
**Author:** Hertzan D. Alquizola II  
**Purpose:** Longitudinal correlation analysis of seasonal PM10 variation and asthma prevalence  
in school-age children (5–12), with socioeconomic confounders.

---
## How to use this notebook
Run each cell **top to bottom** using `Shift + Enter`.  
Cells marked with **`# TODO`** are places where YOU need to check/update a file path or variable name.

## Step 0: Import Libraries
Run this first. If you get an error, open your terminal and run: `pip install -r requirements.txt`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import STL
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Arial'
sns.set_style('whitegrid')

print('All libraries loaded successfully!')

All libraries loaded successfully!


---
## Step 1: Load the CCHAIN Dataset
Download from: https://data.humdata.org/dataset/project-cchain  
Save the files inside: `../data/raw/cchain/`

The CCHAIN dataset has multiple tables. We mainly need:
- **Health table** — asthma/respiratory case counts by barangay and date
- **Environment table** — air quality / PM data
- **Socioeconomic table** — poverty incidence, household density

Check the CCHAIN documentation to confirm exact filenames after you download.

In [6]:
import pandas as pd

# Real CCHAIN file — the main disease dataset you downloaded
CCHAIN_HEALTH_PATH = '../data/raw/cchain/disease_lgu_disaggregated_totals.csv'

df_health = pd.read_csv(CCHAIN_HEALTH_PATH)

print("ROWS, COLUMNS:", df_health.shape)
print("\nCOLUMN NAMES:")
print(df_health.columns.tolist())
print("\nFIRST 3 ROWS:")
df_health.head(3)

ROWS, COLUMNS: (48735, 13)

COLUMN NAMES:
['uuid', 'freq', 'date', 'source_name', 'source_filename', 'adm3_pcode', 'adm4_pcode', 'disease_icd10_code', 'disease_common_name', 'sex', 'age_group', 'case_total', 'death_total']

FIRST 3 ROWS:


,uuid,freq,date,source_name,source_filename,adm3_pcode,adm4_pcode,disease_icd10_code,disease_common_name,sex,age_group,case_total,death_total
0,DLGUT000000,M,2022-01-01,FHSIS-LGU-DISAGG,CdO_2022_Morbidity,PH104305000,NaN,A01,TYPHOID FEVER,F,50-59,0,NaN
1,DLGUT000001,M,2022-01-01,FHSIS-LGU-DISAGG,CdO_2022_Morbidity,PH104305000,NaN,A01,TYPHOID FEVER,M,50-59,1,NaN
2,DLGUT000002,M,2022-02-01,FHSIS-LGU-DISAGG,CdO_2022_Morbidity,PH104305000,NaN,A01,TYPHOID FEVER,F,10-19,0,NaN


In [7]:
# What respiratory diseases are in here?
resp = df_health[df_health['disease_common_name'].str.contains('ASTHMA|RESPIRATORY|PNEUMONIA|BRONCH', case=False, na=False)]
print("RESPIRATORY-RELATED DISEASES FOUND:")
print(resp['disease_common_name'].unique())

print("\nALL AGE GROUPS available:")
print(sorted(df_health['age_group'].dropna().unique()))

RESPIRATORY-RELATED DISEASES FOUND:
<StringArray>
['ASTHMA']
Length: 1, dtype: str

ALL AGE GROUPS available:
['0-4', '10-19', '20-29', '30-39', '40-49', '5-10', '5-9', '50-59', '60+']


In [8]:
# Find Davao's rows (Davao City pcode starts with PH1102...)
print("CITIES (source files) in this dataset:")
print(df_health['source_filename'].unique())

CITIES (source files) in this dataset:
<StringArray>
[                                'CdO_2022_Morbidity',
                                 'CdO_2023_Morbidity',
                  'CdO_2016_INFECTIOUS_DISEASE_clean',
                  'CdO_2017_INFECTIOUS_DISEASE_clean',
                  'CdO_2018_INFECTIOUS_DISEASE_clean',
                  'CdO_2019_INFECTIOUS_DISEASE_clean',
                  'CdO_2020_INFECTIOUS_DISEASE_clean',
                  'CdO_2021_INFECTIOUS_DISEASE_clean',
                  'CdO_2022_INFECTIOUS_DISEASE_clean',
                                       'Muntinlupa_2',
                                       'Muntinlupa_3',
                                       'Muntinlupa_1',
                                       'Muntinlupa_4',
             'Palayan_Tabulation_Morbidity_2020-2022',
                   'Tacloban_Infectious_disease_2022',
 'Zamboanga_City_Morbidity_Report_Consolidation_2021',
                          'Zamboanga_PIDSR_2013-2022']
Length: 17, 

In [9]:
# Check the OTHER two disease files for Davao
for fname in ['disease_pidsr_totals.csv', 'disease_fhsis_totals.csv']:
    d = pd.read_csv(f'../data/raw/cchain/{fname}')
    print(f"\n===== {fname} =====")
    print("Columns:", d.columns.tolist())
    # look for any Davao reference
    for col in d.columns:
        if d[col].astype(str).str.contains('Davao|davao|PH1102|112402', na=False).any():
            print(f"  DAVAO FOUND in column '{col}'")
    # show what cities/sources exist
    src_col = 'source_filename' if 'source_filename' in d.columns else d.columns[0]
    print("Sources/sample:", d[src_col].unique()[:20])


===== disease_pidsr_totals.csv =====
Columns: ['uuid', 'freq', 'date', 'source_name', 'source_filename', 'adm3_pcode', 'disease_icd10_code', 'disease_common_name', 'case_total']
  DAVAO FOUND in column 'adm3_pcode'
Sources/sample: <StringArray>
['City-weekly_2008-2022']
Length: 1, dtype: str

===== disease_fhsis_totals.csv =====
Columns: ['uuid', 'freq', 'date', 'source_name', 'source_filename', 'adm3_pcode', 'disease_icd10_code', 'disease_common_name', 'case_total', 'death_total']
Sources/sample: <StringArray>
[                            'FHSIS_Palayan',
                            'FHSIS_Tacloban',
                           'FHSIS_Zamboanga',
                                 'FHSIS_CdO',
 'FHSIS_Navotas_Mortality_2016_to_2018_2022',
          'FHSIS_Navotas_Morbidity_2017_OCR',
         'FHSIS_Navotas_Morbidity_2018_2022',
          'FHSIS_Navotas_Morbidity_2018_OCR',
                    'FHSIS_Muntinlupa_5_OCR']
Length: 9, dtype: str


In [11]:
# Load PIDSR and isolate Davao asthma
pidsr = pd.read_csv('../data/raw/cchain/disease_pidsr_totals.csv')

# Find Davao's exact pcode
davao_codes = pidsr[pidsr['adm3_pcode'].astype(str).str.contains('PH1102|112402', na=False)]['adm3_pcode'].unique()
print("Davao pcode(s) found:", davao_codes)

# Filter: Davao + Asthma
davao_asthma = pidsr[
    (pidsr['adm3_pcode'].isin(davao_codes)) &
    (pidsr['disease_common_name'].str.contains('ASTHMA', case=False, na=False))
]

print("\nDavao asthma rows:", len(davao_asthma))
print("Date range:", davao_asthma['date'].min(), "to", davao_asthma['date'].max())
print("Total asthma cases recorded:", davao_asthma['case_total'].sum())
print("\nSample:")
print(davao_asthma[['date','disease_common_name','case_total']].head(10))

Davao pcode(s) found: <StringArray>
['PH112402000']
Length: 1, dtype: str

Davao asthma rows: 0
Date range: nan to nan
Total asthma cases recorded: 0

Sample:
Empty DataFrame
Columns: [date, disease_common_name, case_total]
Index: []


In [12]:
# 1. What diseases DOES Davao have in PIDSR?
davao_all = pidsr[pidsr['adm3_pcode'] == 'PH112402000']
print("Davao PIDSR rows total:", len(davao_all))
print("Diseases tracked for Davao in PIDSR:")
print(sorted(davao_all['disease_common_name'].unique()))

# 2. Does Davao show up at all in the FHSIS file (the monthly one)?
fhsis = pd.read_csv('../data/raw/cchain/disease_fhsis_totals.csv')
print("\nFHSIS pcodes containing Davao:")
print(fhsis[fhsis['adm3_pcode'].astype(str).str.contains('PH1102|112402', na=False)]['adm3_pcode'].unique())
print("FHSIS source files:", fhsis['source_filename'].unique())

Davao PIDSR rows total: 6240
Diseases tracked for Davao in PIDSR:
['ACUTE BLOODY DIARRHEA', 'CHIKUNGUYA VIRAL DISEASE', 'CHOLERA', 'DENGUE FEVER', 'HEPATITIS A', 'LEPTOSPIROSIS', 'RABIES', 'ROTAVIRAL ENTERITIS', 'TYPHOID FEVER']

FHSIS pcodes containing Davao:
<StringArray>
[]
Length: 0, dtype: str
FHSIS source files: <StringArray>
[                            'FHSIS_Palayan',
                            'FHSIS_Tacloban',
                           'FHSIS_Zamboanga',
                                 'FHSIS_CdO',
 'FHSIS_Navotas_Mortality_2016_to_2018_2022',
          'FHSIS_Navotas_Morbidity_2017_OCR',
         'FHSIS_Navotas_Morbidity_2018_2022',
          'FHSIS_Navotas_Morbidity_2018_OCR',
                    'FHSIS_Muntinlupa_5_OCR']
Length: 9, dtype: str


In [13]:
# Verify CdO asthma in the barangay-level file (the rich one from earlier)
cdo_asthma = df_health[
    (df_health['source_filename'].str.contains('CdO', case=False, na=False)) &
    (df_health['disease_common_name'].str.contains('ASTHMA', case=False, na=False))
]
print("CdO asthma rows:", len(cdo_asthma))
print("Date range:", cdo_asthma['date'].min(), "to", cdo_asthma['date'].max())
print("Total asthma cases:", cdo_asthma['case_total'].sum())
print("\nAge groups present in CdO asthma data:")
print(sorted(cdo_asthma['age_group'].dropna().unique()))
print("\nBarangays (adm4) present:", cdo_asthma['adm4_pcode'].nunique())
print("\nSample rows:")
print(cdo_asthma[['date','age_group','sex','case_total','adm4_pcode']].head(10))

CdO asthma rows: 80
Date range: 2023-04-01 to 2023-08-01
Total asthma cases: 2

Age groups present in CdO asthma data:
['0-4', '10-19', '20-29', '30-39', '40-49', '5-9', '50-59', '60+']

Barangays (adm4) present: 0

Sample rows:
           date age_group sex  case_total adm4_pcode
370  2023-04-01       0-4   F           0        NaN
371  2023-04-01       0-4   M           0        NaN
372  2023-04-01     10-19   F           0        NaN
373  2023-04-01     10-19   M           0        NaN
374  2023-04-01     20-29   F           0        NaN
375  2023-04-01     20-29   M           0        NaN
376  2023-04-01     30-39   F           0        NaN
377  2023-04-01     30-39   M           0        NaN
378  2023-04-01     40-49   F           0        NaN
379  2023-04-01     40-49   M           0        NaN


In [14]:
# Which diseases have REAL data volume, and where?
big = pidsr.groupby(['adm3_pcode','disease_common_name'])['case_total'].sum().reset_index()
big = big[big['case_total'] > 100].sort_values('case_total', ascending=False)
print("Davao =  PH112402000")
print(big[big['adm3_pcode']=='PH112402000'].to_string())

Davao =  PH112402000
     adm3_pcode       disease_common_name  case_total
75  PH112402000              DENGUE FEVER       66192
72  PH112402000     ACUTE BLOODY DIARRHEA        2424
80  PH112402000             TYPHOID FEVER        1516
77  PH112402000             LEPTOSPIROSIS         843
76  PH112402000               HEPATITIS A         307
74  PH112402000                   CHOLERA         235
73  PH112402000  CHIKUNGUYA VIRAL DISEASE         103


---
## Step 2: Filter for Davao City + Relevant Variables
We only want Davao City rows, years 2013–2022.

In [ ]:
# TODO: After Step 1, check the actual column names and update these
# Common CCHAIN column names are 'city', 'year', 'month', 'barangay_code'

CITY_COL  = 'city'       # column that identifies the city
YEAR_COL  = 'year'       # column for year
CITY_NAME = 'Davao'      # how Davao is labeled in the data

# Filter health data
health_davao = df_health[
    (df_health[CITY_COL].str.contains(CITY_NAME, case=False, na=False)) &
    (df_health[YEAR_COL] >= 2013) &
    (df_health[YEAR_COL] <= 2022)
].copy()

# Filter environment data
env_davao = df_env[
    (df_env[CITY_COL].str.contains(CITY_NAME, case=False, na=False)) &
    (df_env[YEAR_COL] >= 2013) &
    (df_env[YEAR_COL] <= 2022)
].copy()

# Filter socioeconomic data
socio_davao = df_socio[
    df_socio[CITY_COL].str.contains(CITY_NAME, case=False, na=False)
].copy()

print(f'Health rows (Davao 2013-2022): {len(health_davao)}')
print(f'Environment rows (Davao 2013-2022): {len(env_davao)}')
print(f'Socioeconomic rows (Davao): {len(socio_davao)}')

# Preview
health_davao.head()

---
## Step 3: Identify and Extract Key Columns
Find the asthma and PM10 columns specifically.

In [ ]:
# Search for asthma-related columns
asthma_cols = [c for c in health_davao.columns if 'asthma' in c.lower() or 'respiratory' in c.lower()]
print('Asthma/respiratory columns found:')
print(asthma_cols)
print()

# Search for PM10-related columns
pm_cols = [c for c in env_davao.columns if 'pm10' in c.lower() or 'pm_10' in c.lower() or 'particulate' in c.lower()]
print('PM10 columns found:')
print(pm_cols)
print()

# Search for poverty/socioeconomic columns
socio_cols = [c for c in socio_davao.columns if 'poverty' in c.lower() or 'income' in c.lower() or 'household' in c.lower()]
print('Socioeconomic columns found:')
print(socio_cols)

---
## Step 4: Merge Datasets
Join health + environment + socioeconomic into one working dataframe.

In [ ]:
# TODO: Update 'barangay_code', 'year', 'month' to match actual column names
MERGE_KEYS = ['barangay_code', 'year', 'month']

# Merge health + environment
df_merged = pd.merge(health_davao, env_davao, on=MERGE_KEYS, how='inner')

# Merge in socioeconomic (usually yearly, not monthly)
df_merged = pd.merge(df_merged, socio_davao, on=['barangay_code', 'year'], how='left')

print(f'Merged dataset shape: {df_merged.shape}')
print(f'Columns: {df_merged.columns.tolist()}')
df_merged.head()

---
## Step 5: Create a Date Column + Seasonal Label

In [ ]:
# Create proper datetime column
df_merged['date'] = pd.to_datetime(df_merged[['year', 'month']].assign(day=1))

# Label dry vs wet season (Philippines: dry = Nov–Apr, wet = May–Oct)
def get_season(month):
    if month in [11, 12, 1, 2, 3, 4]:
        return 'Dry'
    else:
        return 'Wet'

df_merged['season'] = df_merged['month'].apply(get_season)

print('Seasonal distribution:')
print(df_merged['season'].value_counts())
df_merged[['date', 'season']].head(10)

---
## Step 6: Descriptive Statistics

In [ ]:
# TODO: Replace 'pm10' and 'asthma_cases' with your actual column names from Step 3
PM10_COL   = 'pm10'          # PM10 concentration column
ASTHMA_COL = 'asthma_cases'  # Asthma case count column
POVERTY_COL = 'poverty_incidence'

print('=== PM10 Summary ===')
print(df_merged[PM10_COL].describe().round(2))
print()
print('=== Asthma Cases Summary ===')
print(df_merged[ASTHMA_COL].describe().round(2))
print()
print('=== By Season ===')
print(df_merged.groupby('season')[[PM10_COL, ASTHMA_COL]].mean().round(2))

---
## Step 7: Figure 1 — PM10 Time Series with Seasonal Overlay

In [ ]:
# Monthly average PM10 across all barangays
monthly_pm10 = df_merged.groupby('date')[PM10_COL].mean().reset_index()
monthly_pm10 = monthly_pm10.sort_values('date')

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(monthly_pm10['date'], monthly_pm10[PM10_COL],
        color='steelblue', linewidth=1.5, label='Monthly Avg PM10')

# WHO 24-hour guideline line
ax.axhline(y=45, color='red', linestyle='--', linewidth=1.2, label='WHO 24-hr Guideline (45 µg/m³)')
ax.axhline(y=15, color='orange', linestyle='--', linewidth=1.2, label='WHO Annual Guideline (15 µg/m³)')

ax.set_title('Monthly Average PM10 Concentrations — Davao City (2013–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('PM10 Concentration (µg/m³)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../outputs/figures/fig1_pm10_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

---
## Step 8: Figure 2 — Asthma Cases Over Time

In [ ]:
monthly_asthma = df_merged.groupby('date')[ASTHMA_COL].sum().reset_index()
monthly_asthma = monthly_asthma.sort_values('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly_asthma['date'], monthly_asthma[ASTHMA_COL],
       width=20, color='coral', alpha=0.8, label='Monthly Asthma Cases (5-12 yr)')

ax.set_title('Monthly Asthma Cases (Ages 5–12) — Davao City (2013–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Cases')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/fig2_asthma_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## Step 9: Figure 3 — Correlation Scatter Plot (PM10 vs Asthma)

In [ ]:
# Drop rows with missing values in key columns
df_corr = df_merged[[PM10_COL, ASTHMA_COL, 'season']].dropna()

# Calculate correlations
pearson_r, pearson_p  = stats.pearsonr(df_corr[PM10_COL], df_corr[ASTHMA_COL])
spearman_r, spearman_p = stats.spearmanr(df_corr[PM10_COL], df_corr[ASTHMA_COL])

print(f'Pearson r  = {pearson_r:.4f}  (p = {pearson_p:.4f})')
print(f'Spearman r = {spearman_r:.4f}  (p = {spearman_p:.4f})')

if pearson_p < 0.05:
    print('=> Statistically SIGNIFICANT correlation (p < 0.05)')
else:
    print('=> NOT statistically significant at p < 0.05')

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'Dry': 'tomato', 'Wet': 'steelblue'}
for season, group in df_corr.groupby('season'):
    ax.scatter(group[PM10_COL], group[ASTHMA_COL],
               c=colors[season], alpha=0.6, label=season, s=40)

# Regression line
m, b = np.polyfit(df_corr[PM10_COL], df_corr[ASTHMA_COL], 1)
x_line = np.linspace(df_corr[PM10_COL].min(), df_corr[PM10_COL].max(), 100)
ax.plot(x_line, m * x_line + b, 'k--', linewidth=1.5, label='Trend line')

ax.set_title(f'PM10 vs Asthma Cases — Pearson r={pearson_r:.3f}, p={pearson_p:.4f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('PM10 Concentration (µg/m³)')
ax.set_ylabel('Asthma Cases (Ages 5–12)')
ax.legend(title='Season')
plt.tight_layout()
plt.savefig('../outputs/figures/fig3_correlation_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Step 10: Multivariate Regression (PM10 + Socioeconomic Covariates)

In [ ]:
# TODO: Add/remove covariates based on what columns you have
# Common CCHAIN socioeconomic variables: poverty_incidence, household_density

df_reg = df_merged[[ASTHMA_COL, PM10_COL, POVERTY_COL]].dropna()

# Build formula
formula = f'{ASTHMA_COL} ~ {PM10_COL} + {POVERTY_COL}'

model = smf.ols(formula=formula, data=df_reg).fit()
print(model.summary())

---
## Step 11: WHO Threshold Analysis

In [ ]:
WHO_24HR_LIMIT  = 45   # µg/m³
WHO_ANNUAL_LIMIT = 15  # µg/m³

total_days   = len(df_merged[PM10_COL].dropna())
exceed_24hr  = (df_merged[PM10_COL] > WHO_24HR_LIMIT).sum()
exceed_annual = (df_merged[PM10_COL] > WHO_ANNUAL_LIMIT).sum()

print(f'Total observations:          {total_days}')
print(f'Exceeds WHO 24-hr limit (45): {exceed_24hr}  ({100*exceed_24hr/total_days:.1f}%)')
print(f'Exceeds WHO annual limit (15): {exceed_annual}  ({100*exceed_annual/total_days:.1f}%)')

# By season
print()
print('Exceedances by season:')
print(df_merged.groupby('season')[PM10_COL]
      .apply(lambda x: (x > WHO_24HR_LIMIT).mean() * 100)
      .round(1)
      .rename('% days exceeding WHO 24hr limit'))

---
## Step 12: Export Results Table

In [ ]:
results_summary = pd.DataFrame({
    'Metric': ['Pearson r', 'Pearson p-value', 'Spearman r', 'Spearman p-value',
               '% obs exceeding WHO 24hr limit', '% obs exceeding WHO annual limit'],
    'Value': [round(pearson_r, 4), round(pearson_p, 4),
              round(spearman_r, 4), round(spearman_p, 4),
              round(100*exceed_24hr/total_days, 1),
              round(100*exceed_annual/total_days, 1)]
})

results_summary.to_csv('../outputs/tables/results_summary.csv', index=False)
print('Results table saved.')
results_summary

---
## ✅ You're Done with the Core Analysis!
**Outputs saved to:**
- `outputs/figures/` — all 3 figures
- `outputs/tables/results_summary.csv` — your key numbers

**Next steps:**
1. Copy your Pearson/Spearman values into Section 4 of the Word doc
2. Insert figures into the paper
3. Use the regression output for Section 4.4
4. Push everything to GitHub (see README)